# 06 — Combined Strategy

Combine momentum and reversal signals using multiple weighting methods. Demonstrate diversification benefits.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Auto-generate synthetic data if none exists
from statarb.utils import load_config
from statarb.data.storage import DataStore
from experiments.generate_synthetic_data import generate_all

cfg = load_config("../experiments/config.yaml")
data_cfg = cfg.get("data", {})
syn_cfg = cfg.get("synthetic", {})
exchange = data_cfg.get("exchange", "binance")
base_dir = data_cfg.get("base_dir", "../data")

store = DataStore(base_dir=base_dir)
symbols = store.list_symbols(exchange, "1d")
if not symbols:
    print("Generating synthetic data...")
    generate_all(
        n_assets=syn_cfg.get("n_assets", 20),
        n_days=syn_cfg.get("n_days", 1000),
        start_date=syn_cfg.get("start_date", "2021-01-01"),
        seed=syn_cfg.get("seed", 42),
        exchange=exchange,
        base_dir=base_dir,
    )
    symbols = store.list_symbols(exchange, "1d")

prices  = store.build_panel(symbols, exchange, "1d", field="close")
volume  = store.build_volume_panel(symbols, exchange, "1d")

from statarb.data.features import FeatureEngine
fe = FeatureEngine()
returns = fe.log_returns(prices)
vol_ratio = fe.volume_ma_ratio(volume, window=21)

print(f"Loaded: {len(symbols)} assets x {len(prices)} days")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")


In [ ]:
from statarb.signals.momentum import MomentumSignals
from statarb.signals.reversal import ReversalSignals
from statarb.signals.activity import ActivityFilter
from statarb.backtest.execution import ExecutionModel
from statarb.backtest.weighting import StrategyWeighting
from statarb.evaluation.metrics import PerformanceMetrics
from experiments._utils import run_bt, backtest_signals

mom = MomentumSignals()
rev = ReversalSignals()
act = ActivityFilter()
em  = ExecutionModel(market_order_cost=0.0020)
pm  = PerformanceMetrics()
sw  = StrategyWeighting()


## Build Component Signals

In [ ]:
raw_signals = {
    "mom_6_1":       mom.momentum_6_1(returns),
    "mom_3_1":       mom.momentum_3_1(returns),
    "ts_mom_63d":    mom.time_series_momentum(returns, lookback=63),
    "sharpe_mom":    mom.sharpe_momentum(returns, lookback=63),
    "reversal_5d":   rev.weekly_reversal(returns),
    "vol_adj_rev":   rev.vol_adjusted_reversal(returns),
}

# Activity-gate and rank
ranked = {
    name: fe.cross_sectional_rank(act.activity_gate(sig, volume, min_ratio=0.5))
    for name, sig in raw_signals.items()
}

# Get individual return streams
strategy_returns = {}
for name, sig in ranked.items():
    net_rets, _ = run_bt(sig, returns, em)
    strategy_returns[name] = net_rets

returns_df = pd.DataFrame(strategy_returns).dropna()
print(f"Individual strategy return DataFrame: {returns_df.shape}")
print(f"Date range: {returns_df.index[0].date()} → {returns_df.index[-1].date()}")


## Individual Strategy Performance

In [ ]:
indiv_summary = {}
for name, rets in strategy_returns.items():
    clean = rets.dropna()
    indiv_summary[name] = {
        "sharpe": pm.sharpe_ratio(clean, periods_per_year=365),
        "ann_ret": pm.annualized_return(clean) * 100,
        "max_dd":  pm.max_drawdown(clean) * 100,
        "win_rate": pm.win_rate(clean) * 100,
    }
indiv_df = pd.DataFrame(indiv_summary).T
print(indiv_df.round(2).to_string())


## Combination Methods

In [ ]:
combined_rets = {}
for method in ["equal", "inverse_vol", "sharpe", "min_variance"]:
    try:
        combined = sw.combine(returns_df, method=method, lookback=63)
        combined_rets[f"combined_{method}"] = combined
        clean = combined.dropna()
        sharpe = pm.sharpe_ratio(clean, periods_per_year=365)
        print(f"  {method:20s}: Sharpe {sharpe:.2f}")
    except Exception as e:
        print(f"  {method}: failed ({e})")


## Diversification Benefit

In [ ]:
avg_individual = indiv_df["sharpe"].mean()
equal_combined = pm.sharpe_ratio(combined_rets.get("combined_equal", pd.Series()).dropna(),
                                   periods_per_year=365)

print(f"Average individual Sharpe:    {avg_individual:.2f}")
print(f"Equal-weight combined Sharpe: {equal_combined:.2f}")
benefit = equal_combined - avg_individual
print(f"Diversification benefit:      +{benefit:.2f} Sharpe units")

# Correlation between individual strategies
fig, ax = plt.subplots(figsize=(8, 6))
import seaborn as sns
corr = returns_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", center=0, cmap="coolwarm",
            square=True, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Strategy Return Correlations", fontweight="bold")
plt.tight_layout()
plt.show()
print("Low correlations → diversification benefit is real, not luck.")


## Combined Strategy Cumulative Returns

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
# Individual strategies (thin, faded)
for name, rets in strategy_returns.items():
    cum = (1 + rets.fillna(0)).cumprod() - 1
    ax.plot(cum.index, cum.values * 100, linewidth=0.8, alpha=0.3)
# Combined (bold)
for name, rets in combined_rets.items():
    cum = (1 + rets.fillna(0)).cumprod() - 1
    ax.plot(cum.index, cum.values * 100, linewidth=2, label=name)
ax.set_title("Individual (faded) vs Combined (bold) Cumulative Returns", fontweight="bold")
ax.set_ylabel("Cumulative Return (%)")
ax.axhline(0, color="black", linewidth=0.6)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()
